# Dimensions, bases, and units

Every quantity in axiom carries a `Dimension`. Three things are kept distinct, because
collapsing them is how the parent repo ended up with a whole `finance/` package:

| | Example | On mismatch |
|---|---|---|
| **Dimension** | currency vs time vs outcome | raise, always |
| **Unit** | USD vs EUR; day vs week | convert if a conversion is registered, with a ledger line; raise if not |
| **Scope** | this population, this window | never auto-resolved; needs a stated assumption (`TransferPlan`, Phase 6) |

This notebook covers the first two rows.

In [ ]:
from fractions import Fraction

from axiom.core import (
    BASES,
    D,
    UNITS,
    BaseRegistry,
    Dimension,
    DimensionError,
    UndeclaredBaseError,
    UnitConversionError,
    UnitSystem,
    dimensionless,
)

## The base set is declarable

`BASES` ships four bases — `time`, `currency`, `outcome`, `entity` — and a domain declares
whatever else it needs. `D` is the same registry; `D.currency` reads better in model code.

In [ ]:
print(list(BASES))
print(D.currency, D.time, D.outcome, D.entity)

In [ ]:
mass = BASES.declare("mass", symbol="M")   # agronomy, pharmacology
print(mass, "|", BASES.declarations())     # non-default declarations travel with a saved analysis

A base that has not been declared is an error naming the base, not a silent new dimension.

In [ ]:
try:
    Dimension(exponents={"temperature": 1})
except UndeclaredBaseError as e:
    print(type(e).__name__, "->", e)

## Algebra

Exponents are `Fraction`s, not ints, because a standard deviation is the square root of a
variance and intervals are everywhere in this package. Zero exponents vanish, so equality
and hashing are structural.

In [ ]:
outcome_per_dose = D.outcome / D.currency
rate = D.outcome / D.time
variance = outcome_per_dose**2

print("outcome per unit dose:", outcome_per_dose)
print("its variance:         ", variance)
print("sd = sqrt(variance):  ", variance.root(2), "==", outcome_per_dose, "->", variance.root(2) == outcome_per_dose)
print("x / x is dimensionless:", (rate / rate) == dimensionless(), (rate / rate).is_dimensionless)
print("fractional powers:     ", D.currency ** Fraction(1, 3))

`require_equal` and `require_dimensionless` are what the expression-tree checker (Phase 1b)
calls at every node; the error names both sides and the node it was checking.

In [ ]:
try:
    D.currency.require_equal(D.time, context="Add(term_1, term_2)")
except DimensionError as e:
    print(e)

try:
    D.currency.require_dimensionless(context="the argument of log()")
except DimensionError as e:
    print(e)

## A `Dimension` is a `Spec`

So it serializes, hashes, and round-trips like everything else (see `02-specs-and-hashing`).

In [ ]:
s = outcome_per_dose.to_json()
print(s)
print(Dimension.from_json(s) == outcome_per_dose, outcome_per_dose.content_hash()[:16])

## Units and conversions

A `UnitSystem` attaches units of measure to bases and registers conversions *within* a
dimension. `UNITS` is the process-global instance; you can also build a private one.
Conversions compose: registering USD→EUR and EUR→GBP makes USD→GBP available.

In [ ]:
UNITS.declare("USD", "currency")
UNITS.declare("EUR", "currency")
UNITS.declare("GBP", "currency")
UNITS.declare("day", "time")
UNITS.declare("week", "time")

UNITS.register("USD", "EUR", Fraction(9, 10))
UNITS.register("EUR", "GBP", Fraction(17, 20))
UNITS.register("week", "day", 7)

print(UNITS.factor("USD", "GBP"), "|", UNITS.dimension_of("week"))

Every conversion performed returns the value **and** a `LedgerLine`. That line is the
provenance rule (CLAUDE.md rule 4) made concrete: the analysis ledger records that a number
crossed a unit boundary and by what factor.

In [ ]:
value, line = UNITS.convert(1000.0, "USD", "GBP")
print(value)
print(line.kind, "|", line.statement)
print(line.detail)

Cross-dimension conversion is refused (that is the *dimension* row of the table), and a
missing conversion is an error rather than a guess (the *unit* row).

In [ ]:
UNITS.declare("JPY", "currency")

for src, dst in [("USD", "day"), ("USD", "JPY")]:
    try:
        UNITS.convert(1.0, src, dst)
    except (DimensionError, UnitConversionError) as e:
        print(f"{src} -> {dst}: {type(e).__name__}: {e}")

A private registry and unit system are useful in tests and in adapters that should not
touch the global declarations.

In [ ]:
private_bases = BaseRegistry()
private_units = UnitSystem()
private_units.declare("kg", "mass")
private_units.declare("g", "mass")
private_units.register("kg", "g", 1000)
print(private_units.convert(2.5, "kg", "g")[0], "|", private_units.units(), "|", private_units.conversions())
print("private registry knows only the defaults:", list(private_bases))